In [1]:
from typing import Optional, Any, List


class Graph:
    """
    Graph class
    """
    def __init__(self):
        self._graph = {}

    def add_vertex(self, vertex: str, data: Optional[Any]=None) -> None:
        """
        Adds a vertex to the graph
        :param vertex: the vertex name
        :param data: data associated with the vertex
        """
        if vertex not in self._graph:
            self._graph[vertex] = {'data': data, 'neighbors': {}}

    def add_edge(self, vertex1: str, vertex2: str, data: Optional[Any]=None) -> None:
        """
        Adds an edge to the graph
        :param vertex1: vertex1 key
        :param vertex2: vertex2 key
        :param data: the data associated with the vertex
        """
        if not vertex1 in self._graph or not vertex2 in self._graph:
            raise ValueError("The vertexes do not exist")
        self._graph[vertex1]['neighbors'][vertex2] = data

    def get_neighbors(self, vertex) -> List[str]:
        """
        Get the list of vertex neighbors
        :param vertex: the vertex to query
        :return: the list of neighbor vertexes
        """
        if vertex in self._graph:
            return list(self._graph[vertex]['neighbors'].keys())
        else:
            return []

    def get_vertex_data(self, vertex: str) -> Optional[Any]:
        """
        Gets  vertex associated data
        :param vertex: the vertex name
        :return: the vertex data
        """
        if self.vertex_exists(vertex):
            return self._graph[vertex]['data']
        else:
            return None

    def get_edge_data(self, vertex1: str, vertex2: str) -> Optional[Any]:
        """
        Gets the vertexes edge data
        :param vertex1: the vertex1 name
        :param vertex2: the vertex2 name
        :return: vertexes edge data
        """
        if self.edge_exists(vertex1, vertex2):
            return self._graph[vertex1]['neighbors'][vertex2]
        raise ValueError("The edge does not exist")

    def print_graph(self) -> None:
        """
        Prints the graph
        """
        for vertex, data in self._graph.items():
            print("Vertex:", vertex)
            print("Data:", data['data'])
            print("Neighbors:", data['neighbors'])
            print("")

    def vertex_exists(self, vertex: str) -> bool:
        """
        If contains a vertex
        :param vertex: the vertex name
        :return: boolean
        """
        return vertex in self._graph

    def edge_exists(self, vertex1: str, vertex2: str) -> bool:
        """
        If contains an edge
        :param vertex1: the vertex1 name
        :param vertex2: the vertex2 name
        :return: boolean
        """
        return vertex1 in self._graph and vertex2 in self._graph[vertex1]['neighbors']

In [2]:
graph = Graph()

Cargar datos en el grafo

In [6]:
from tqdm import tqdm

def fill_graph(graph:Graph,non_directed:bool = False):
    index_title_dict = {}

    with open("../data/vertexes.txt", "r", encoding="utf-8") as file:
        for line in file:
            index = int(line.split(" ")[0])
            title = " ".join(line[:-1].split(" ")[1:])
            index_title_dict[index] = title
            graph.add_vertex(title)

    with open("../data/edges.txt", "r", encoding="utf-8") as file:
        for line in tqdm(file, desc="Loading edges", total=101409330):
            art1 = int(line[:-1].split(" ")[0])
            art2 = int(line[:-1].split(" ")[1])
            q = int(line[:-1].split(" ")[2])
            if art1 in index_title_dict and art2 in index_title_dict:
                graph.add_edge(index_title_dict[art1], index_title_dict[art2], q)
                if(non_directed): graph.add_edge(index_title_dict[art2], index_title_dict[art1], q)

    del index_title_dict

In [ ]:
import pickle

fill_graph(graph)

with open("../data/graph.pkl", "wb") as file:
    pickle.dump(graph, file)

In [3]:
import pickle

with open("../data/graph.pkl", "rb") as file:
    graph = pickle.load(file)

Cantidad de componenetes conexas

In [8]:
from collections import deque

non_directed_graph = Graph()
fill_graph(non_directed_graph,non_directed=True)

def count_connected_components(graph:Graph):
    visited = set()
    cant = 0

    def bfs(start):
        queue = deque()
        queue.append(start)
        while queue:
            vertex = queue.popleft()
            for neighbor in graph.get_neighbors(vertex):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)

    for vertex in graph._graph.keys():
        if vertex not in visited:
            visited.add(vertex)
            cant += 1
            bfs(vertex)
    
    return cant

conex_components = count_connected_components(non_directed_graph)

if(conex_components == 1):
    print("El grafo es debilmente conexo")
else:
    print("El grafo no es debilmente conexo")
    print("Total de componentes conexas: ", conex_components)

non_directed_graph = None


Loading edges: 100%|█████████▉| 101409095/101409330 [04:55<00:00, 342642.42it/s]


El grafo no es debilmente conexo
Total de componentes conexas:  32665


Wikipedia Race

In [4]:
def search_path(v1:str,v2:str,previous:list):
    path = []
    current = v2
    while current != v1:
        path.append(current)
        current = previous[current]
    path.append(v1)
    path.reverse()
    return path

def print_path(path):
    for i in range(len(path)-1):
        print(path[i],end=" -> ")
    print(path[-1])

In [19]:
#Elegir articulos
while True:
    v1 = input("Ingrese el nombre del primer articulo: ")
    v2 = input("Ingrese el nombre del segundo articulo: ")
    if v1 in graph._graph and v2 in graph._graph:
        break
    print("Uno de los articulos ingresados no existe.")

In [20]:
from collections import deque

def BFS(graph:Graph, start:str, end:str)->dict:
    if start == end:
        return [start]
    queue = deque()
    visited = set()
    previous = {}
    queue.append(start)
    visited.add(start)
    while(len(queue) != 0):
        vertex = queue.popleft()
        for neighbor in graph.get_neighbors(vertex):
            if neighbor == end:
                previous[neighbor] = vertex
                return previous
            if neighbor not in visited:
                queue.append(neighbor)
                visited.add(neighbor)
                previous[neighbor] = vertex
    return None

previous = BFS(graph, v1 , v2)
if previous is None:
        print("Not connected.")
else:
    path = search_path(v1,v2,previous)
    print_path(path)



Samsung -> Renault -> Fernando Alonso -> Pixar


Caminos más y menos originales

In [17]:

def camino_original(graph:Graph, start:str, end:str,min_paths:dict)->None:
    queue = deque()
    visited = set()
    previous = {}
    len_min_path = float('inf')
    found_min_path = False
    queue.append((start,0,0))
    visited.add(start)
    while(len(queue) != 0):
        vertex,current_len,links = queue.popleft()
        for neighbor in graph.get_neighbors(vertex):
            if current_len > len_min_path:
                continue
            if neighbor == end:
                if(not found_min_path):
                    len_min_path = current_len
                    found_min_path = True

                previous[neighbor] = vertex
                path = search_path(start,end,previous)
                min_paths[links//len_min_path] = path
            if neighbor not in visited:
                queue.append((neighbor,current_len+1,links+graph.get_edge_data(vertex,neighbor)))
                visited.add(neighbor)
                previous[neighbor] = vertex
    return None

min_paths_dict = {}
camino_original(graph, v1 , v2, min_paths_dict)
more_original_path = min_paths_dict[min(min_paths_dict.keys())]
less_original_path = min_paths_dict[max(min_paths_dict.keys())]

print("Camino mas original (con un promedio de ",min(min_paths_dict.keys())," enlaces por articulo):")
print_path(more_original_path)
print()
print("Camino menos original (con un promedio de ",max(min_paths_dict.keys())," enlaces por articulo):")
print_path(less_original_path)

del min_paths_dict


Camino mas original (con un promedio de  1  enlaces por articulo):
Samsung -> Apple -> TikTok -> Pixar

Camino menos original (con un promedio de  8  enlaces por articulo):
Samsung -> Apple -> Steve Jobs -> Pixar


Camino más largo del grafo

In [5]:
from collections import deque

def find_max_path(graph,start,max_path_len:int)->list:
    queue = deque()
    visited = set()
    previous = {}
    queue.append((start,0))
    visited.add(start)
    max_path = []
    while len(queue) != 0:
        vertex,size = queue.popleft()
        
        for neighbor in graph.get_neighbors(vertex):
            if neighbor not in visited:
                visited.add(neighbor)
                previous[neighbor] = vertex
                queue.append((neighbor,size+1))

        if size > max_path_len:
            max_path_len = size
            max_path = search_path(start,vertex,previous)

    return max_path,max_path_len
    

In [ ]:
import random
def neighbors_not_in_path(vertex:str,path:list[str],graph:Graph)->bool:
    for neighbor in graph.get_neighbors(vertex):
        if neighbor in path:
            return False
    return True     

def find_longest_path(graph:Graph,cant_iteraciones:int):
    longest_path = []
    len_longest_path = 0
    max_path_len = 0
    for _ in range(cant_iteraciones):
        max_path,max_path_len = find_max_path(graph,random.choice(list(graph._graph.keys())),max_path_len)
        if max_path_len > len_longest_path:
            longest_path = max_path
            len_longest_path = max_path_len
    start_vertex = longest_path[0]
    flag = True
    while flag:
        for vertex in graph._graph.keys():
            if start_vertex in graph.get_neighbors(vertex) and vertex not in longest_path:
                if neighbors_not_in_path(vertex,longest_path,graph):
                    longest_path.insert(0,vertex)
                    max_path_len +=1
                    flag = True
                    break
            flag = False
    return longest_path,len_longest_path

longest_path,size = find_longest_path(graph,100)
print("Camino mas largo aproximado (de tamaño",size,"):")
print_path(longest_path)

Camino mas largo aproximado (de tamaño 38 ):
Azerbaiyán en los Juegos Olímpicos de Pekín 2022 -> Control de autoridades -> Tesauro -> Unesco -> Suiza -> Roger Federer -> Francisco Clavet -> Masters de Cincinnati 2000 -> Masters de Cincinnati 1999 -> Masters de Cincinnati 1998 -> Masters de Cincinnati 1997 -> Masters de Cincinnati 1996 -> Masters de Cincinnati 1995 -> Masters de Cincinnati 1994 -> Masters de Cincinnati 1993 -> Masters de Cincinnati 1992 -> Masters de Cincinnati 1991 -> Masters de Cincinnati 1990 -> Masters de Cincinnati 1989 -> Masters de Cincinnati 1988 -> Masters de Cincinnati 1987 -> Masters de Cincinnati 1986 -> Masters de Cincinnati 1985 -> Masters de Cincinnati 1984 -> Masters de Cincinnati 1983 -> Masters de Cincinnati 1982 -> Masters de Cincinnati 1981 -> Masters de Cincinnati 1980 -> Masters de Cincinnati 1979 -> Masters de Cincinnati 1978 -> Masters de Cincinnati 1977 -> Masters de Cincinnati 1976 -> Masters de Cincinnati 1975 -> Masters de Cincinnati 1974 -> 

Artículos que no llevan a ningun otro, que no se puede llegar desde ningun otro y con links correspondidos

In [9]:
data_dict = {
    "vertex_without_neighbors": {"count": 0, "vertices": []},
    "corresponding_vertex": {"count": 0, "pairs": []},
    "unreachable_vertex": {"count": 0, "vertices": []}
}

in_degrees = {vertex:0 for vertex in graph._graph.keys()}
out_degrees = {vertex:0 for vertex in graph._graph.keys()}

for vertex in graph._graph.keys():
    for neighbor in graph.get_neighbors(vertex):
        in_degrees[neighbor] += 1
        out_degrees[vertex] += 1

for vertex in graph._graph.keys():
    for neighbor in graph.get_neighbors(vertex):
        if graph.edge_exists(neighbor,vertex):
            data_dict["corresponding_vertex"]["count"] += 1
            if(data_dict["corresponding_vertex"]["count"] <= 5):
                data_dict["corresponding_vertex"]["pairs"].append((vertex,neighbor))

    if out_degrees[vertex] == 0:
        if graph.get_neighbors(vertex) == []:
            data_dict["vertex_without_neighbors"]["count"] += 1
            if(data_dict["vertex_without_neighbors"]["count"] <= 5):
                data_dict["vertex_without_neighbors"]["vertices"].append(vertex)

    if in_degrees[vertex] == 0:
        data_dict["unreachable_vertex"]["count"] += 1
        if(data_dict["unreachable_vertex"]["count"] <= 5):
            data_dict["unreachable_vertex"]["vertices"].append(vertex)

del in_degrees
del out_degrees

In [13]:
print("Vertices sin vecinos: ",data_dict["vertex_without_neighbors"]["count"])
print("Primeros 5 encontrados:",data_dict["vertex_without_neighbors"]["vertices"],"\n")
print("Vertices correspondientes: ",data_dict["corresponding_vertex"]["count"])
print("Primeros 5 encontrados:",data_dict["corresponding_vertex"]["pairs"],"\n")
print("Vertices no alcanzables: ",data_dict["unreachable_vertex"]["count"])
print("Primeros 5 encontrados:",data_dict["unreachable_vertex"]["vertices"],"\n")

Vertices sin vecinos:  32793
Primeros 5 encontrados: ['Blind date', "Heaven's Gate", 'Live & Rare', 'Liubov Yegórova', 'Código del Trabajo de Chile (desambiguación)'] 

Vertices correspondientes:  2999882
Primeros 5 encontrados: [('Pictilabrus viridis', 'Pictilabrus'), ('Stenoma cremastis', 'Stenoma'), ('Excoecaria', 'Hippomaneae'), ('Excoecaria', 'Hippomaninae'), ('Melissa De Sousa', '30 Years to Life')] 

Vertices no alcanzables:  2958256
Primeros 5 encontrados: ['Pickpockets: Maestros del robo', 'Aeropuerto Internacional Capitán FAP Carlos Martínez de Pinillos', 'Richard C. Lewontin', 'Federacion Eslovena de Futbol', 'Condado de Renville (Dakota del Norte)'] 



Label Propagation

In [ ]:
import random
from collections import defaultdict

def label_propagation(graph:Graph,cant_iteraciones:int,etiquetas_fijas:List[str]=[])->dict:
    labels = {vertex:vertex for vertex in graph._graph.keys()}
    
    for _ in range(cant_iteraciones):
        stable = True
        vertices = list(graph._graph.keys())
        random.shuffle(vertices)
        for vertex in vertices:
            if vertex in etiquetas_fijas:
                continue
            neighbors = graph.get_neighbors(vertex)
            if neighbors:
                label_count = defaultdict(int)
                neighbor_labels = [labels[neighbor] for neighbor in neighbors]
                for label in neighbor_labels:
                    label_count[label] += 1
                max_count = max(label_count.values())
                for label in neighbor_labels:
                    if label_count[label] == max_count:
                        if label in etiquetas_fijas:
                            most_common_label = label
                            break
                        most_common_label = label
                if labels[vertex] != most_common_label:
                    labels[vertex] = most_common_label
                    stable = False
        if stable:
            break

    if not stable:
        print("El algoritmo no convergio")

    return labels


In [ ]:
#Comumnidades con etiquetas fijas

etiquetas_fijas = ["Javier Milei", "LGA 1200", "Guarana", "Escuela de la Bauhaus"]
labels = label_propagation(graph, 10, etiquetas_fijas)

comunidades ={
    "Javier Milei": [vertex for vertex in labels.keys() if labels[vertex] == "Javier Milei"],
    "LGA 1200": [vertex for vertex in labels if labels[vertex] == "LGA 1200"],
    "Guarana Antartica": [vertex for vertex in labels if labels[vertex] == "Guarana"],
    "Escuela de la Bauhaus": [vertex for vertex in labels if labels[vertex] == "Escuela de la Bauhaus"],
}

for comunidad in comunidades:
    print("Comunidad ",comunidad,":")
    print(comunidades[comunidad][0:10])
    print()


Comunidad  Javier Milei :
['Javier Milei']

Comunidad  LGA 1200 :
['LGA 1200']

Comunidad  Guaraná Antartica :
[]

Comunidad  Escuela de la Bauhaus :
['Escuela de la Bauhaus', 'Bauhaus (desambiguación)']



In [ ]:
from collections import defaultdict

#Comunidades sin etiquetas fijas

#Busco las 5 comunidades mas grandes
comunidades = []
for _ in range(5):
    label_count = defaultdict(int)
    for label in labels.key():
        label_count[label] += 1
    max_count = max(label_count.values())
    for label in labels.key():


Centralidad del grafo (Random Walks)

In [ ]:
import random

def centralidad_random_walks(graph:Graph, alpha=0.85, max_iterations=10, tolerance=1e-6):
    # Inicialización
    vertices = list(graph._graph.keys())
    len_graph = len(vertices)  
    ranks = {vertex: 1 / len_graph for vertex in vertices}  # Probabilidad inicial uniforme
    new_ranks = {vertex: 0 for vertex in vertices}
    
    for _ in range(max_iterations):
        # Reiniciar new_ranks
        for vertex in vertices:
            new_ranks[vertex] = (1 - alpha) / len_graph
        
        
        for vertex in vertices:
            neighbors = graph.get_neighbors(vertex)
            if neighbors:  
                contribution = ranks[vertex] / len(neighbors)
                for neighbor in neighbors:
                    new_ranks[neighbor] += alpha * contribution
        
        # Verificar convergencia
        diff = sum(abs(new_ranks[vertex] - ranks[vertex]) for vertex in vertices)
        if diff < tolerance:
            break
        
        # Actualizar ranks
        ranks, new_ranks = new_ranks, ranks  
    
    return ranks

result = centralidad_random_walks(graph)

In [32]:
def print_results(result:dict):
    print("Estadísticas de Centralidad:\n")
    centralities = list(result.values())
    print("Promedio:", np.mean(centralities))
    print("Mediana:", np.median(centralities))
    print("Máximo:", max(centralities))
    print("Mínimo:", min(centralities))

    top_nodes = sorted(result.items(), key=lambda x: x[1], reverse=True)[:10]
    print("\n10 nodos más centrales:")
    for node, rank in top_nodes:
        print(f"Node {node}: {rank:.6f}")

In [ ]:
import numpy as np

print_results(result)

# plt.hist(centralities, bins=50, color='blue', alpha=0.7)
# plt.xlabel("Centralidad")
# plt.ylabel("Frecuencia")
# plt.title("Distribución de Centralidad")
# plt.show()


Estadísticas de Centralidad:

Promedio: 2.508316849410919e-07
Mediana: 3.789360083184033e-08
Máximo: 0.04151430838800686
Mínimo: 3.789360083184033e-08

10 nodos más centrales:
Node Control de autoridades: 0.041514
Node Wayback Machine: 0.022148
Node Tesauro: 0.018100
Node Gemeinsame Normdatei: 0.011986
Node Library of Congress Control Number: 0.009995
Node Biblioteca Nacional de Israel: 0.009679
Node Fichero de Autoridades Virtual Internacional: 0.008825
Node ISBN: 0.008583
Node Wikimedia Commons: 0.006638
Node Biblioteca Nacional de Francia: 0.006565


Centralidad del grafo (Betweenness)

In [33]:
from collections import defaultdict

def betweenness_centrality(graph:Graph,cant_vertex:int):
    vertices = random.sample(list(graph._graph.keys()), cant_vertex)
    centralidad = defaultdict(float)

    for v1 in vertices:
        for v2 in vertices:
            if v1 != v2:
                previous = BFS(graph,v1,v2)
                if previous is not None:
                    path = search_path(v1,v2,previous)
                    for vertex in path:
                        centralidad[vertex] += 1 / len(path)
    #Normalizar
    len_graph = len(vertices) * (len(vertices) - 1)
    for vertex in centralidad:
        centralidad[vertex] /= len_graph
    
    return centralidad

result = betweenness_centrality(graph,10)

Estimación de clustering del grafo

In [34]:
import numpy as np

print_results(result)

Estadísticas de Centralidad:

Promedio: 0.0038461538461538464
Mediana: 0.0012345679012345679
Máximo: 0.010987654320987654
Mínimo: 0.0011111111111111111

10 nodos más centrales:
Node Francia: 0.010988
Node 24 Horas de Le Mans: 0.010988
Node Kunos Simulazioni: 0.010988
Node Ferrari Virtual Academy: 0.010988
Node Ferrari Virtual Race: 0.010988
Node Synetic: 0.010988
Node Crash Time II: 0.010988
Node Puerco (desambiguación): 0.001235
Node Sus scrofa domestica: 0.001235
Node Ser más feo que Picio: 0.001235


In [ ]:
import networkx as nx
import random

def estimate_clustering(graph:Graph, cant_vertex:int):
    vertices = random.sample(list(graph._graph.keys()), cant_vertex)

    coeficientes = {vertex: nx.clustering(graph, vertex) for vertex in vertices}

    v_max = max(coeficientes, key=coeficientes.get)
    v_min = min(coeficientes, key=coeficientes.get)
    
    return {
        "maximo": (v_max, coeficientes[v_max]),
        "minimo": (v_min, coeficientes[v_min]),
        "promedio": sum(coeficientes.values()) / len(coeficientes)
    }

result = estimate_clustering(graph, 1000)

print("Coeficiente de clustering máximo:", result["maximo"])
print("Coeficiente de clustering mínimo:", result["minimo"])
print("Coeficiente de clustering promedio:", result["promedio"])

In [ ]:
import heapq

def dijkstra(graph:Graph, node:str)->None:
    visited = {}
    dist = {}
    previous = {}
    for vertex in graph._graph.keys():
        dist[vertex] = float('inf')
        previous[vertex] = None
        visited[vertex] = False
    dist[node] = 0  
    heap = []
    heapq.heappush(heap,(0,node))
    while len(heap) != 0 :
        _,vertex = heapq.heappop(heap)
        if visited[vertex]:
            continue
        visited[vertex] = True
        for neighbor in graph.get_neighbors(vertex):
            if not visited[neighbor] and dist[neighbor] > dist[vertex] + graph.get_edge_data(vertex,neighbor):
                dist[neighbor] = dist[vertex] + graph.get_edge_data(vertex,neighbor)
                previous[neighbor] = vertex
            if(not visited[neighbor]): heapq.heappush(heap,(dist[neighbor],neighbor))
               
    return previous,dist


